In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# 使用 OpenAI API 密钥设置一个名为 OPENAI_API_KEY 的环境变量。安装 Python 库
!pip install llama-index
!pip install llama-index-core

!pip install llama-index-llms-openai-like

!pip install llama-index-llms-dashscope
!pip install llama-index-embeddings-dashscope


In [7]:
# from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# documents = SimpleDirectoryReader("data").load_data()
# index = VectorStoreIndex.from_documents(documents)
# query_engine = index.as_query_engine()
# response = query_engine.query("怎么休事假？")
# print(response)

# 配置通义千问大模型和文本向量模型
import os
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.dashscope import DashScopeEmbedding, DashScopeTextEmbeddingModels

Settings.llm = OpenAILike(
    model="qwen-plus",
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    is_chat_model=True
)

Settings.embed_model = DashScopeEmbedding(
    model_name=DashScopeTextEmbeddingModels.TEXT_EMBEDDING_V3,
    embed_batch_size=6,
    embed_input_length=8192
)

documents = SimpleDirectoryReader("data").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()
response = query_engine.query("怎么休事假？")
print(response)


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
2025-11-23 22:59:48,490 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


申请事假需提前向直属主管提出并获得批准，紧急情况可事后补办手续。事假为无薪假，按日扣除相应工资。每月事假原则上不超过3天，全年累计不超过15天；超过此限的特殊情况，需经人力资源部及公司领导审批。


In [19]:
from llama_index.core import PromptTemplate
query_gen_str = """\
系统角色设定:
你是一个专业的问题改写助手。你的任务是将用户的原始问题扩充为一个更完整、更全面的问题。

规则：
1. 将可能的歧义、相关概念和上下文信息整合到一个完整的问题中
2. 使用括号对歧义概念进行补充说明
3. 添加关键的限定词和修饰语
4. 确保改写后的问题清晰且语义完整
5. 对于模糊概念，在括号中列举主要可能性

原始问题:
{query}

请生成一个综合的改写问题，确保：
- 包含原始问题的核心意图
- 涵盖可能的歧义解释
- 使用清晰的逻辑关系词连接不同方面
- 必要时使用括号补充说明

输出格式：
[问题] - 改写后的问题
"""
query_gen_prompt = PromptTemplate(query_gen_str)
query = "怎么休事假？"
formatted_prompt = query_gen_prompt.format(query=query) 
print(formatted_prompt)

question_rewritting_response = Settings.llm.predict(query_gen_prompt, query=query)
print(question_rewritting_response)
query_response = query_engine.query(question_rewritting_response)
print(query_response)


系统角色设定:
你是一个专业的问题改写助手。你的任务是将用户的原始问题扩充为一个更完整、更全面的问题。

规则：
1. 将可能的歧义、相关概念和上下文信息整合到一个完整的问题中
2. 使用括号对歧义概念进行补充说明
3. 添加关键的限定词和修饰语
4. 确保改写后的问题清晰且语义完整
5. 对于模糊概念，在括号中列举主要可能性

原始问题:
怎么休事假？

请生成一个综合的改写问题，确保：
- 包含原始问题的核心意图
- 涵盖可能的歧义解释
- 使用清晰的逻辑关系词连接不同方面
- 必要时使用括号补充说明

输出格式：
[问题] - 改写后的问题



2025-11-23 23:37:23,675 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


[问题] - 如何申请事假（包括但不限于个人事务、家庭事务、突发情况等），具体流程是什么（如是否需要提前申请、提交书面/电子申请、提供证明材料等），事假期间的薪资待遇如何计算（是否为带薪事假或无薪事假），不同单位或企业（如国企、私企、事业单位、外企等）在事假规定上可能存在哪些差异，以及相关法律法规（如《劳动法》或地方性规定）对此有何要求？


2025-11-23 23:37:31,225 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


申请事假需根据公司规定进行，员工因处理个人事务、家庭事务或突发情况等必须本人处理的事项，可申请事假。事假需提前申请并获得直属主管批准；如遇紧急情况，可事后补办相关手续。

所有休假申请应通过公司OA系统或书面提交《员工请假申请表》。事假为无薪假，按实际请假天数扣除相应工资。每月事假原则上不超过3天，全年累计不超过15天；超过此限制的特殊情况，须经人力资源部及公司领导审批。

关于不同单位的事假差异，国企、事业单位通常有较为统一的管理制度，事假天数和审批流程可能更严格，并受地方性人事管理政策影响；私企和外企则多依据自身规章制度执行，灵活性较高，但不得低于法定标准。部分企业可能对事假次数和时长设置更严格的限制，或要求提供相关事由说明。

根据《中华人民共和国劳动法》及相关法规，法律未明确规定事假的具体天数和待遇，属于企业自主管理范畴。因此，事假是否批准、如何计算薪资，主要依据劳动合同约定及用人单位依法制定的规章制度。但用人单位在制定事假政策时，不得违反公平原则或变相侵害劳动者合法权益。


In [ ]:
from llama_index.core.indices.query.query_transform.base import StepDecomposeQueryTransform
from llama_index.core.query_engine import MultiStepQueryEngine

step_decompose_transform = StepDecomposeQueryTransform(llm=Settings.llm, verbose=True)

query_engine = index.as_query_engine()
multi_query_engine = MultiStepQueryEngine(
    query_engine=query_engine,
    query_transform=step_decompose_transform,
    index_summary="这是一个公司手册",
)
response = multi_query_engine.query("怎么休事假？")
print(response)

2025-11-23 23:56:31,574 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


> Current query: 怎么休事假？
> New query: 怎么休事假？


2025-11-23 23:56:34,356 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-23 23:56:34,984 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


> Current query: 怎么休事假？
> New query: 怎么申请事假？


2025-11-23 23:56:37,647 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-23 23:56:38,126 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


> Current query: 怎么休事假？
> New query: 怎么申请事假？


2025-11-23 23:56:41,257 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-23 23:56:45,796 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


申请事假需提前提交申请并获得直属主管批准；如遇紧急情况，可事后补办手续。事假为无薪假，按日扣除相应工资。每月事假原则上不超过3天，全年累计不超过15天，特殊情况需经人力资源部及公司领导审批。所有请假需通过公司OA系统或书面填写《员工请假申请表》提交。


In [36]:
from llama_index.core.indices.query.query_transform.base import (
    HyDEQueryTransform,
)
from llama_index.core.query_engine import TransformQueryEngine
test_query_str = "怎么休事假？"
# run query with HyDE query transform
hyde = HyDEQueryTransform(include_original=True)
query_engine = index.as_query_engine()
print(query_engine.query(test_query_str))
query_engine2 = index.as_query_engine(streaming=True,similarity_top_k=5)
print(query_engine2.query(test_query_str))
query_bundle = hyde(test_query_str)
print(query_bundle)
final_query_engine = TransformQueryEngine(query_engine, query_transform=hyde)

print(final_query_engine.query(test_query_str))

2025-11-24 00:01:48,029 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


申请事假需提前向直属主管提出并获得批准，紧急情况可事后补办手续。事假为无薪假，按日扣除相应工资。每月事假原则上不超过3天，全年累计不超过15天；超过此限或有特殊情况的，需经人力资源部及公司领导审批。所有请假均需通过公司OA系统或书面提交《员工请假申请表》办理。


2025-11-24 00:01:48,703 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


休事假需提前申请并获得直属主管批准，紧急情况可事后补办手续。事假为无薪假，按日扣除相应工资。每月事假原则上不超过3天，全年累计不超过15天；特殊情况需经人力资源部及公司领导审批。所有请假均需通过公司OA系统或书面提交《员工请假申请表》。


2025-11-24 00:02:03,035 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


怎么休事假？


2025-11-24 00:02:11,026 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-24 00:02:14,520 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


申请事假需提前向直属主管提出并获得批准，紧急情况可事后补办手续。事假为无薪假，按日扣除相应工资。每月事假原则上不超过3天，全年累计不超过15天；超过规定天数的特殊情况，需经人力资源部及公司领导审批。所有请假均需通过公司OA系统或书面提交《员工请假申请表》办理。


In [ ]:
import os
from dashscope import Assistants
from dashscope.assistants import Message 

LEAVE_ROUTER_INSTRUCTION = """你是一个专业的员工休假政策问题分类器。
请根据用户的提问内容，判断其咨询的休假类型。

请严格根据以下类别进行判断，输出结果只能是下列选项中的一个，不得添加任何解释、标点或额外信息：

- 年休假（用户询问年假天数、何时能休、是否可以拆分、未休补偿等）
- 病假（用户询问病假申请流程、需要什么证明、工资如何发放等）
- 事假（用户询问事假申请、审批流程、是否带薪等）
- 婚假（用户询问婚假天数、有效期、是否包含周末等）
- 产假或陪产假（用户询问产假/陪产假时长、生育津贴、能否延长等）
- 丧假（用户询问丧假天数、适用亲属范围等）
- 法定节假日（用户询问春节、国庆等节假日安排、调休等）
- 其他休假（如育儿假、哺乳假、工伤假等未明确列出的假期类型）
- 非休假问题（用户的问题与请假、休假政策完全无关，例如考勤打卡、薪资、离职等）

示例输出：
年休假
"""

router_agent = Assistants.create(
    model="qwen-plus",  # 推荐使用 qwen-plus：效果均衡，支持长上下文，适合理解复杂语义
    name='休假问题分类器',
    description='负责将员工关于休假政策的咨询自动分类到对应假期类型，用于路由至专业问答模块。',
    instructions=LEAVE_ROUTER_INSTRUCTION
)

# 思考： 为什么要用框架？


**数据层**  
- 如何支持多种数据源（如本地文件、云存储、数据库、API）的统一接入？  
- 多种文件类型（PDF、DOCX、CSV、PPT）能否被自动识别并正确解析？  
- 面对编码错误、损坏文件或加密文档，解析过程是否具备容错能力？  
- 目录路径“data”写死，是否支持通过配置动态指定？  
- 是否支持递归读取子目录以及按规则排除特定文件或文件夹？  
- 用户能否自定义文件解析逻辑，例如为特定格式注册自己的解析器？  

>  说明：LlamaIndex 提供 `SimpleDirectoryReader` 支持常见格式、递归读取、文件过滤（`required_exts`、`exclude_hidden`），并可通过继承 `BaseReader` 实现自定义解析器。

---

**索引层**  
- 向量存储后端（如 FAISS、Chroma、Pinecone、Weaviate、Milvus）是否可灵活切换？  
- 文档切分策略是否可配置，例如按固定长度、语义边界或段落分割？  
- 是否支持元数据（如来源路径、作者、时间）嵌入索引以用于过滤？  
- 索引能否持久化保存，避免每次重复构建？  
- 新增文档时是否支持增量索引更新而非全量重建？  
- 是否可以组合使用向量检索与关键词检索实现混合搜索？  

>  说明：LlamaIndex 支持多种 `VectorStore` 集成、`NodeParser`（如 `SentenceSplitter`）可配置切分、支持元数据过滤、索引持久化（`storage_context.persist()`）、增量添加文档、以及通过 `AutoMergingRetriever` 或 `BM25Retriever + VectorIndexRetriever` 实现混合检索。

---

**查询层**  
- 查询是否支持流式输出，以便前端实时展示生成内容？  
- 是否能结合对话历史实现多轮问答和上下文理解？  
- 不同类型的问题是否可以路由到不同的查询引擎（如规则引擎 vs 向量检索）？  
- 是否具备意图识别能力，判断用户问题是政策咨询、流程操作还是其他类别？  
- 如何处理模糊或歧义查询，是否支持查询重写或澄清提问？  

>  说明：支持流式响应（`StreamingResponse`）、`ChatEngine` 支持对话历史、可通过 `Tool` 或自定义 `RouterQueryEngine` 实现路由、支持 `SubQuestionQueryEngine` 进行查询分解与重写。

---

**输出层**  
- 返回的 `response` 对象是否能转换为标准 JSON 格式以便 API 集成？  
- 是否可以在答案中附带引用来源、原始文本片段和置信度信息？  
- 输出结果是否可序列化并跨网络传输？  
- 是否提供结构化输出能力，例如返回固定 schema 的 JSON 数据？  
- 生成内容是否经过安全检查，防止恶意 prompt 注入或泄露训练数据？  

>  说明：可通过 `response.get_response()` 提取结构信息、引用节点可通过 `source_nodes` 获取、支持 `PydanticOutputParser` 生成结构化输出。但安全过滤需外部介入。

---

**业务层**  
- 系统是否适配具体业务场景，如 HR 政策查询、IT 支持、员工自助服务？  
- 不同用户角色（员工、HR、管理员）是否能看到不同范围或精度的结果？  
- 业务规则（如事假需审批、最多3天）是否能与检索结果融合输出？  
- 用户对回答不满意时，是否有反馈机制来收集问题并优化知识库？  
- 高频未命中问题是否可被记录，用于后续知识补全？  

>  说明：LlamaIndex 不直接实现权限控制或反馈闭环，但可通过外部逻辑集成实现“按角色过滤文档”、“记录 query 日志”等，框架本身支持元数据过滤和可扩展性。

---

**工程层**  
- 文件不存在、格式不支持、解析失败等情况是否有异常捕获和处理？  
- 大量文档加载或高并发查询时，系统性能是否可控？是否支持异步处理？  
- 是否记录关键日志用于调试和链路追踪？  
- 是否具备基本监控指标（如查询延迟、命中率、token 消耗）？  
- 整个流程是否可集成到 CI/CD 流水线中，支持自动化部署？  

>  说明：异常为标准 Python 异常，可捕获；支持异步加载文档和异步查询；日志使用标准 logging；token 使用可通过回调获取；工程化集成依赖外部系统，但组件本身可编程组装。

---

**安全层**  
- 加载的文档是否可能包含敏感信息（如身份证号、薪资）？读取时是否需权限控制？  
- 在文本切分和向量化过程中，是否会无意中暴露隐私内容？  
- 是否支持对敏感字段进行自动脱敏或过滤？  
- 输出内容是否经过合规性校验，避免生成违法或不当言论？  
- 整个系统是否符合企业级安全审计和数据保护要求？  

> 说明：LlamaIndex 本身不提供脱敏、内容过滤、权限校验等安全机制，但这些问题是合理的设计考量，可在其基础上由应用层实现。

---

**扩展层**  
- 是否允许第三方开发者注册自定义数据连接器或解析插件？  
- 是否支持接入图像、表格、扫描件等多模态内容的理解能力？  
- 大语言模型是否可自由替换，支持本地部署模型或私有化服务？  
- 是否提供开放接口或 SDK，供外部系统集成调用？  
- 是否具备扩展能力以支持未来新增的数据源、索引类型或查询模式？  

>  说明：LlamaIndex 设计为高度可扩展，支持自定义 Reader、NodeParser、Retriever、LLM、Embedding、Tool 等；支持 HuggingFace、Ollama、本地模型；提供 Python SDK，适合集成。
